# Introduction to h3xplorer

In [1]:
from pathlib import Path

import polars as pl

from h3xplorer import core

In [2]:
data_dir = Path.cwd().parent / "tests" / "fixture_data"
xys = core._import_dataset(data_dir / "xy.parquet")
xys = xys.with_columns(pl.Series("population", [300, 200, 1000, 100, 600]))
latlons = core._read_xy_dataset(xys, "x", "y", 27700)
df_hex_refs, hex_refs = core._get_hexagon_refs_for_points(latlons, 2)
display(df_hex_refs)
display(hex_refs)

2025-10-08 20:30:20,526 [INFO] Loaded in dataset: xy.parquet
2025-10-08 20:30:20,533 [INFO] Converting points to EPSG:4326
2025-10-08 20:30:20,571 [INFO] Point conversion complete
2025-10-08 20:30:20,572 [INFO] Getting hexagon references
Converting points to h3 references: 100%|██████████| 5/5 [00:00<00:00, 29831.47it/s]
2025-10-08 20:30:20,580 [INFO] Hexagon references retrieved


id,population,lon,lat,h3_ref
i64,i64,f64,f64,i64
1,300,-3.448079,51.689822,585914353279041535
2,200,-3.448079,51.689822,585914353279041535
3,1000,-0.494362,53.487243,585912704011599871
4,100,-0.128329,51.503992,585913253767413759
5,600,-2.001375,51.249166,585914353279041535


{585912704011599871, 585913253767413759, 585914353279041535}

In [3]:
hexes = core._get_hexagon_polygons(hex_refs)
df_agg = core._groupby_ref_col(df_hex_refs, pop_sum={"column": "population", "agg": "sum"})
gdf = core._join_pldf_to_gdf(df_agg, hexes)
gdf

2025-10-08 20:30:20,718 [INFO] Creating hexagon polygons table
Converting h3 references to polygons: 100%|██████████| 3/3 [00:00<00:00, 11254.84it/s]
2025-10-08 20:30:20,723 [INFO] aggregating data input to spatial areas


,geometry,pop_sum
h3_ref,,
585912704011599871,"POLYGON ((-0.21914 52.14168, 2.10131 52.51713,...",1000
585913253767413759,"POLYGON ((-0.21914 52.14168, -0.63109 50.60191...",100
585914353279041535,"POLYGON ((-2.90653 50.14229, -0.63109 50.60191...",1100


In [4]:
core._plot_polygon_data(gdf, "pop_sum")

Map(basemap_style=<CartoBasemap.DarkMatter: 'https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json'…